[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/07_logistic_regression/91_assignment_logistic_regression.ipynb)

# Assignment 91 — Logistic Regression

Nhiệm vụ: dự đoán một thí sinh **có được nhận vào đại học hay không** dựa trên điểm 2 bài thi (bài tập kinh điển của khoá Andrew Ng). Học trước bài [01](../../../notebooks/07_logistic_regression/01_logistic_regression.ipynb) và [02](../../../notebooks/07_logistic_regression/02_from_scratch_and_optimization.ipynb).

### Hướng dẫn làm bài
- Dùng NumPy, vector hoá (không vòng lặp trên dữ liệu).
- Viết code trong `### START CODE HERE ###` … `### END CODE HERE ###`, chạy cell kiểm tra.

### Chú ý: Advanced Optimization
Bài này dùng `scipy.optimize.minimize` (BFGS) thay cho gradient descent: không cần chọn learning rate, hội tụ nhanh. Bạn chỉ cần cung cấp hàm **cost** và hàm **gradient**.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

from ml_course.checks import check_close, check_equal, check_greater, check_true
from ml_course.utils import numerical_gradient, relative_error, repo_path

## 1. Trực quan hoá dữ liệu

In [ ]:
data = np.loadtxt(repo_path("data", "sample", "ex4_exam_scores.txt"), delimiter=",")
X, y = data[:, :2], data[:, 2]
print(X[:5], y[:5])


def plot_data(X, y, xlabel, ylabel, label_pos, label_neg, ax=None):
    ax = ax or plt.gca()
    pos, neg = y == 1, y == 0
    ax.scatter(X[pos, 0], X[pos, 1], s=30, c="darkblue", marker="+", label=label_pos)
    ax.scatter(X[neg, 0], X[neg, 1], s=30, c="gold", marker="o", edgecolors="k", linewidths=0.4, label=label_neg)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper right", frameon=True)


plot_data(X, y, "Exam 1 score", "Exam 2 score", "Admitted", "Not admitted")
plt.show()

## 2. Thực hiện bài toán

### 2.1 Hàm sigmoid

$$h_\theta(x) = g(\theta^\top x), \qquad g(z) = \frac{1}{1 + e^{-z}}$$

**Bài tập:** viết hàm `sigmoid` (nhận số hoặc mảng NumPy).

In [ ]:
def sigmoid(z):
    ### START CODE HERE ### (≈ 1 dòng)
    g = 1 / (1 + np.exp(-z))
    ### END CODE HERE ###
    return g


x_val = np.linspace(-10, 10, 1000)
plt.plot(x_val, sigmoid(x_val))
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.show()

In [ ]:
check_close(sigmoid(0), 0.5, message="sigmoid(0)")
check_close(sigmoid(np.array([-100.0, 100.0])), [0, 1], tol=1e-9, message="sigmoid ở hai đầu")

### 2.2 Cost function và gradient

Thêm cột 1 (bias) vào X:

In [ ]:
m = len(y)
X_b = np.hstack([np.ones((m, 1)), X])

**Bài tập:** viết `cost_function(theta, X, y)`.

$$h = g(X\theta), \qquad J(\theta) = -\frac{1}{m}\left[y^\top\log(h) + (1 - y)^\top\log(1 - h)\right]$$

**Chú ý:** do làm tròn số, `sigmoid` có thể trả về đúng 0 hoặc 1 khiến `log(0)` không xác định → cộng một số rất nhỏ `eps = 1e-15` vào trong log.

In [ ]:
def cost_function(theta, X, y):
    eps = 1e-15
    ### START CODE HERE ### (≈ 2 dòng)
    h = sigmoid(X @ theta)
    J = -(y @ np.log(h + eps) + (1 - y) @ np.log(1 - h + eps)) / len(y)
    ### END CODE HERE ###
    return J


print("Cost tại θ = 0:", cost_function(np.zeros(3), X_b, y))

In [ ]:
check_close(cost_function(np.zeros(3), X_b, y), 0.6931471805599452, tol=1e-6, message="J(0) = ln 2")
check_close(cost_function(np.array([-24.0, 0.2, 0.2]), X_b, y), 0.21833019382659785, tol=1e-6, message="J tại θ thử")

**Đầu ra kỳ vọng:** `Cost tại θ = 0: 0.693147`

**Bài tập:** viết `gradient_func(theta, X, y)`:

$$\nabla J(\theta) = \frac{1}{m}X^\top\left(g(X\theta) - y\right)$$

In [ ]:
def gradient_func(theta, X, y):
    ### START CODE HERE ### (≈ 2 dòng)
    h = sigmoid(X @ theta)
    grad = X.T @ (h - y) / len(y)
    ### END CODE HERE ###
    return grad


print("Gradient tại θ = 0:", gradient_func(np.zeros(3), X_b, y))

In [ ]:
check_close(gradient_func(np.zeros(3), X_b, y), [-0.1, -12.00921659, -11.26284221], tol=1e-6, message="gradient tại θ = 0")
t = np.array([-24.0, 0.2, 0.2])
check_true(relative_error(gradient_func(t, X_b, y), numerical_gradient(lambda th: cost_function(th, X_b, y), t.copy())) < 1e-6,
           "gradient check")

**Đầu ra kỳ vọng:** `[-0.1  -12.0092  -11.2628]`

## 3. Tìm θ bằng `scipy.optimize.minimize`

**Bài tập:** gọi `minimize(cost_function, x0, args=(X_b, y), jac=gradient_func, method="BFGS")` rồi lấy `.x`.

In [ ]:
initial_theta = np.zeros(3)
### START CODE HERE ### (≈ 2 dòng)
result = minimize(cost_function, initial_theta, args=(X_b, y), jac=gradient_func, method="BFGS")
theta = result.x
### END CODE HERE ###
print("θ tối ưu:", theta)
print("Cost tại θ tối ưu:", cost_function(theta, X_b, y))

In [ ]:
check_close(cost_function(theta, X_b, y), 0.20349770158947478, tol=1e-5, message="cost tối ưu ≈ 0.2035")

**Đầu ra kỳ vọng:** cost ≈ `0.2035`, θ ≈ `[-25.16, 0.206, 0.201]`

Vẽ **decision boundary** $\theta_0 + \theta_1 x_1 + \theta_2 x_2 = 0 \Rightarrow x_2 = -(\theta_0 + \theta_1 x_1)/\theta_2$:

In [ ]:
x1_line = np.array([X[:, 0].min(), X[:, 0].max()])
plot_data(X, y, "Exam 1 score", "Exam 2 score", "Admitted", "Not admitted")
plt.plot(x1_line, -(theta[0] + theta[1] * x1_line) / theta[2], "r", label="decision boundary")
plt.legend()
plt.show()

## 4. Đánh giá Logistic Regression

**Bài tập:** viết `predict(theta, X)` trả về nhãn 0/1 (ngưỡng 0.5); tính xác suất được nhận của thí sinh có điểm **45** và **85**; tính accuracy trên tập train.

In [ ]:
def predict(theta, X):
    ### START CODE HERE ### (≈ 1 dòng)
    p = (sigmoid(X @ theta) >= 0.5).astype(int)
    ### END CODE HERE ###
    return p


### START CODE HERE ### (≈ 2 dòng)
prob_45_85 = sigmoid(np.array([1, 45, 85]) @ theta)
train_accuracy = np.mean(predict(theta, X_b) == y)
### END CODE HERE ###
print(f"Xác suất được nhận (45, 85): {prob_45_85:.3f}")
print(f"Accuracy train: {train_accuracy:.2%}")

In [ ]:
check_close(prob_45_85, 0.776, tol=2e-3, message="P(admit | 45, 85) ≈ 0.776")
check_close(train_accuracy, 0.89, message="accuracy train = 89%")

**Đầu ra kỳ vọng:** xác suất ≈ `0.776`, accuracy train = `89%`

### Tổng kết

- Cài đặt sigmoid, cross-entropy cost và gradient (vector hoá), kiểm tra bằng gradient check.
- Dùng advanced optimization (BFGS) qua `scipy.optimize.minimize`.
- Vẽ decision boundary tuyến tính và đánh giá model.

---
<sub>Bài gốc: *Programming Assignment: Logistic Regression* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License), dựa trên bài tập của Andrew Ng. Đã thêm kiểm tra tự động, gradient check, bỏ phụ thuộc seaborn.</sub>